In [1]:
## Pinecone VectorDB

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv


def _load_repo_dotenv() -> None:
    """Load `.env` next to `pyproject.toml` so we never pick up another folder's .env."""
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        env, marker = d / ".env", d / "pyproject.toml"
        if env.is_file() and marker.is_file():
            load_dotenv(env, override=True)
            return
    load_dotenv(override=True)


_load_repo_dotenv()


In [3]:
def _require(name: str) -> str:
    v = os.getenv(name)
    if v is None:
        raise RuntimeError(f"Missing {name} in project root .env")
    v = str(v).strip().strip('"').strip("'")
    if not v:
        raise RuntimeError(f"Missing {name} in project root .env")
    return v


os.environ["PINECONE_API_KEY"] = _require("PINECONE_API_KEY")
os.environ["OPENAI_API_KEY"] = _require("OPENAI_API_KEY")

_pk = os.environ["PINECONE_API_KEY"]
if not (_pk.startswith("pcsk_") or _pk.startswith("pckey_")):
    raise ValueError(
        "PINECONE_API_KEY must start with pcsk_ or pckey_. Copy the full key from the Pinecone console (API Keys)."
    )

In [4]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

In [5]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small', dimensions=1024)

d:\Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import os
from pinecone import Pinecone, ServerlessSpec

# Rebuild client each time this cell runs (kernel can keep a stale `pc` after you edit .env)
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "rag"  # change if desired
if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimensions=1024,
        metric = "cosine",
        spec = ServerlessSpec(cloud = 'aws', region = 'us-east-1')
    )

index = pc.Index(index_name)

In [7]:
index

In [8]:
from langchain_pinecone import PineconeVectorStore
vector_store  = PineconeVectorStore(index, embeddings)

In [9]:
vector_store

In [10]:
## Create Documents
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [11]:
vector_store.add_documents(documents = documents)

['f608ca3e-827d-481b-97f9-496fb03a374d',
 '14cfbc92-9f5e-492a-83e8-29cd3906331e',
 '08ff329c-91c0-4613-8922-5b9d7f6b5151',
 '9fbbe2d3-253b-4a6f-80c2-47a697518079',
 '130eb56c-86db-49cc-8eb0-5200afdad4bd',
 '81e63a08-edc5-4836-bedd-57480bb1719d',
 '94327033-3e19-4a2a-9d4e-4e644f812e4d',
 'c93d302a-6275-43a8-b8e5-5aa07fdcc771',
 '4f148008-3219-49f9-bd8a-6347e697a520',
 '878a7f5c-1d6e-4a55-950b-ee7d963bc1f5']

In [13]:
# Query Directly
results = vector_store.similarity_search(
    "Langchain provides abstractions to make working with LLMs easy",
    k = 2,
    filter = {"source": "tweet"},
)
for res in results:
    print(f"*  {res.page_content} [{res.metadata}]")

*  LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]
*  Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]


In [14]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k =1,filter= {"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:.2f}] {res.page_content}[{res.metadata}]")

* [SIM=0.57] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.[{'source': 'news'}]


In [16]:
### Retriever
retriever=vector_store.as_retriever(
  search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.5},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='9fbbe2d3-253b-4a6f-80c2-47a697518079', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]